# MCP Client with Gemma 3

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

#MODEL_ID = "google/gemma-3-1b-pt" #transformers.models.gemma3.modeling_gemma3.Gemma3ForCausalLM
#MODEL_ID = "google/gemma-3-1b-it" #transformers.models.gemma3.modeling_gemma3.Gemma3ForCausalLM
MODEL_ID = "google/gemma-3-4b-it" #transformers.models.gemma3.modeling_gemma3.Gemma3ForConditionalGeneration
#MODEL_ID = "google/gemma-3-12b-it" #transformers.models.gemma3.modeling_gemma3.Gemma3ForConditionalGeneration



In [2]:
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=MODEL_ID,
    dtype="auto",
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
type(model)

transformers.models.gemma3.modeling_gemma3.Gemma3ForConditionalGeneration

In [4]:
model.device

device(type='mps', index=0)

In [5]:
import torch
device = "cpu"
if torch.backends.mps.is_built():
    device = torch.device("mps")  # for mac use
device

device(type='mps')

In [16]:
system_prompt = """
You are an assistent that calls tools using MCP (Model Context Protocol).
If the user asks for a calculation, respond ONLY with ONE JSON-RPC request acording the following examples:

User: Multipy 23 with 19.
Assistent:
{
    "jsonrpc": "2.0",
    "method": "calculator.multiply",
    "params": {"x":23, "y": 19},
    "id": "req-123"
}

User: Multipy 8 with 5.
Assistent:
{
    "jsonrpc": "2.0",
    "method": "calculator.multiply",
    "params": {"x":8, "y": 5},
    "id": "req-124"
}
"""

In [17]:
user_input = "Multiply 6 with 7."

In [18]:
prompt = f"{system_prompt}\nUser: {user_input}\nAssistent:"

In [19]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids.to(device)

In [20]:
inputs.input_ids[0]

tensor([     2,    107,   3048,    659,    614,   6361,    533,    600,   9139,
          6436,   1699, 106284,    568,   4968,  17605,  42065,    769,    107,
          2859,    506,   2430,  19565,    573,    496,  14988, 236764,   8932,
         44917,    607,  33015,  10434, 236772,  84026,   2864,   1226,   2424,
           506,   2269,   8698, 236787,    108,   2887, 236787,  10112,  40493,
        236743, 236778, 236800,    607, 236743, 236770, 236819, 236761,    107,
          7721,  13182, 236787,    107, 236782,    107,    140, 236775,   3723,
         82831,   1083,    623, 236778, 236761, 236771,    827,    107,    140,
        236775,   8889,   1083,    623,  47552, 236761,  64545,    827,    107,
           140, 236775,   6160,   1083,   3714, 236781,   1083, 236778, 236800,
        236764,    623, 236762,   1083, 236743, 236770, 236819,   1263,    107,
           140, 236775,    547,   1083,    623,   5749, 236772, 236770, 236778,
        236800, 236775,    107, 236783, 

In [21]:
" ".join(inputs.tokens())

'<bos> \n You ▁are ▁an ▁assist ent ▁that ▁calls ▁tools ▁using ▁MCP ▁( Model ▁Context ▁Protocol ). \n If ▁the ▁user ▁asks ▁for ▁a ▁calculation , ▁respond ▁ONLY ▁with ▁ONE ▁JSON - RPC ▁request ▁ac ording ▁the ▁following ▁examples : \n\n User : ▁Mult ipy ▁ 2 3 ▁with ▁ 1 9 . \n Ass istent : \n { \n ▁▁▁▁ " json rpc ": ▁" 2 . 0 ", \n ▁▁▁▁ " method ": ▁" calculator . multiply ", \n ▁▁▁▁ " params ": ▁{" x ": 2 3 , ▁" y ": ▁ 1 9 }, \n ▁▁▁▁ " id ": ▁" req - 1 2 3 " \n } \n\n User : ▁Mult ipy ▁ 8 ▁with ▁ 5 . \n Ass istent : \n { \n ▁▁▁▁ " json rpc ": ▁" 2 . 0 ", \n ▁▁▁▁ " method ": ▁" calculator . multiply ", \n ▁▁▁▁ " params ": ▁{" x ": 8 , ▁" y ": ▁ 5 }, \n ▁▁▁▁ " id ": ▁" req - 1 2 4 " \n } \n\n User : ▁Multiply ▁ 6 ▁with ▁ 7 . \n Ass istent :'

In [22]:
#model.config
model.generation_config

GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": [
    1,
    106
  ],
  "pad_token_id": 0,
  "top_k": 64,
  "top_p": 0.95
}

In [23]:
# Generate
generate_ids = model.generate(input_ids, max_new_tokens=225)

In [24]:
#tokenizer.batch_decode(generate_ids)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

'\nYou are an assistent that calls tools using MCP (Model Context Protocol).\nIf the user asks for a calculation, respond ONLY with ONE JSON-RPC request acording the following examples:\n\nUser: Multipy 23 with 19.\nAssistent:\n{\n    "jsonrpc": "2.0",\n    "method": "calculator.multiply",\n    "params": {"x":23, "y": 19},\n    "id": "req-123"\n}\n\nUser: Multipy 8 with 5.\nAssistent:\n{\n    "jsonrpc": "2.0",\n    "method": "calculator.multiply",\n    "params": {"x":8, "y": 5},\n    "id": "req-124"\n}\n\nUser: Multiply 6 with 7.\nAssistent:\n{\n    "jsonrpc": "2.0",\n    "method": "calculator.multiply",\n    "params": {"x":6, "y": 7},\n    "id": "req-125"\n}\n'

In [52]:
###### new System prompt #######
system_prompt = """
You are an assistant that produces strictly valid JSON-RPC 2.0 requests for an MCP (Model Context Protocol) server. 
Do not add explanations, comments, or text outside of JSON. 

"""
mcp_server = ["calculator.multiply","calculator.add", "weather.get"]
user_prompt = "How is the weather tomorrow in Berlin?"
prompt = f"""{system_prompt}
You are knowing the following MCP server: {", ".join(mcp_server)} 
Use the one that matches best to the user question. Generate ony one answer to the user question. Stop after first JSON was generated. 
Do not add more user questions. 
User: {user_prompt}
Assistent:"""

In [54]:
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids.to(device)
generate_ids = model.generate(input_ids, max_new_tokens=225)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

'\nYou are an assistant that produces strictly valid JSON-RPC 2.0 requests for an MCP (Model Context Protocol) server. \nDo not add explanations, comments, or text outside of JSON. \n\n\nYou are knowing the following MCP server: calculator.multiply, calculator.add, weather.get \nUse the one that matches best to the user question. Generate ony one answer to the user question. Stop after first JSON was generated. \nDo not add more user questions. \nUser: How is the weather tomorrow in Berlin?\nAssistent:\n```json\n{"method": "weather.get", "params": {"location": "Berlin", "date": "tomorrow"}}\n```\n'